Probelm Statement:
What if embeddings can store mathematical structure as well. Say 9.. somehow it has stored the meaning of 9 (in absolute math terms), such that when we actually do 9 + 9, the mathematical mearning part of the embeddings is itself 18! When we do 9*9.. it becomes 81! How much can we push? Can we descibe whole mathematics and all mathematical operations using this? Of course we need some space for alphabets/words as well, for that we can use 32 existing spaces, and add this new concept into new ones (that are appended)!

 Structured Mathematical Encoding
 To make vector arithmetic perform real math, we can't rely on random initialization. We have to custom-design how a number is represented in vector space.   
 1. Direct Value & Sign Tracking: The first few dimensions hold the raw scalar value, log magnitude ($\log\vert{}n\vert{}$), a sign indicator, and a zero flag. This allows linear addition to happen naturally via vector addition.

2. Log-Space Multiplication: Multiplying two numbers directly in linear space is difficult for neural network vectors, but multiplication in log space becomes simple addition ($\log(a \times b) = \log(a) + \log(b)$).   

 3. Modular Clock Ratios: To handle periodic properties and cyclical bounds, numbers are encoded using coprime clock limits ([7, 11, 13, 17, 19, 23]) mapped to sine and cosine components (mirroring Chinese Remainder Theorem principles).

In [31]:
import math, random, re, itertools
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F

random.seed(0); torch.manual_seed(0)

CLOCKS = [7, 11, 13, 17, 19, 23]       # no shared factors
CLOCK_LIMIT = math.prod(CLOCKS)        # clocks identify numbers up to ~7.4 million

def encode(n):
    """Number -> vector: [value, log|n|, sign, is_zero, cos/sin for each clock]"""
    vec = [n, math.log(abs(n)) if n else 0.0, float(np.sign(n)), float(n == 0)]
    for T in CLOCKS:
        angle = 2 * math.pi * (n % T) / T
        vec += [math.cos(angle), math.sin(angle)]
    return np.array(vec)

def add(x, y):
    out = encode(int(round(x[0] + y[0])))        # values add; log form is recalculated (a known limit)
    for i in range(4, len(x), 2):                # clocks: adding = turning the hands
        out[i]     = x[i] * y[i] - x[i + 1] * y[i + 1]
        out[i + 1] = x[i + 1] * y[i] + x[i] * y[i + 1]
    return out

def negate(x):
    y = x.copy()
    y[0], y[2] = -x[0], -x[2]            # flip value and sign
    y[5::2] = -x[5::2]                   # turn clock hands backwards
    return y

def sub(x, y):
    return add(x, negate(y))

def mul(x, y):
    if x[3] or y[3]:                     # anything times 0 is 0
        return encode(0)
    return encode(int(round(x[2] * y[2] * math.exp(x[1] + y[1])))) # logs add -> numbers multiply

def decode_value(x):
    return int(round(x[0]))


def decode_clocks(x):
    """Read the number back from the clocks only (Chinese Remainder Theorem)."""
    n = 0
    for k, T in enumerate(CLOCKS):
        angle = math.atan2(x[5 + 2 * k], x[4 + 2 * k])
        remainder = round(angle * T / (2 * math.pi)) % T
        rest = CLOCK_LIMIT // T
        n += remainder * rest * pow(rest, -1, T)
    n %= CLOCK_LIMIT
    return n - CLOCK_LIMIT if n >= CLOCK_LIMIT // 2 else n


print("9 + 9 =", decode_value(add(encode(9), encode(9))))
print("9 * 9 =", decode_value(mul(encode(9), encode(9))))
print("9 + 9 (clocks only) =", decode_clocks(add(encode(9), encode(9))))

9 + 9 = 18
9 * 9 = 81
9 + 9 (clocks only) = 18


In [32]:
failures = {"add": 0, "sub": 0, "mul": 0, "clocks": 0}
for a, b in itertools.product(range(-100, 101), repeat=2):
    ea, eb = encode(a), encode(b)
    failures["add"] += decode_value(add(ea, eb)) != a + b
    failures["sub"] += decode_value(sub(ea, eb)) != a - b
    failures["mul"] += decode_value(mul(ea, eb)) != a * b
    failures["clocks"] += decode_clocks(add(ea, eb)) != a + b
print("Every pair from -100 to 100 (40,401 per operation). Failures:", failures)

print("\nMultiplication exactness as numbers grow (computer rounding limit):")
for digits in range(2, 11):
    ok = sum(decode_value(mul(encode(a), encode(b))) == a * b
             for a, b in ((random.randint(10**(digits-1), 10**digits),
                           random.randint(10**(digits-1), 10**digits)) for _ in range(1000)))
    print(f" {digits}-digit x {digits}-digit: {ok / 10:.1f}% exact")

print(f"\nClocks work inside ±{CLOCK_LIMIT // 2}:")
print("  3,000,000 ->", decode_clocks(encode(3_000_000)))
print("  4,000,000 ->", decode_clocks(encode(4_000_000)), "(wraps around, as theory predicts)")

Every pair from -100 to 100 (40,401 per operation). Failures: {'add': 0, 'sub': 0, 'mul': 0, 'clocks': 0}

Multiplication exactness as numbers grow (computer rounding limit):
 2-digit x 2-digit: 100.0% exact
 3-digit x 3-digit: 100.0% exact
 4-digit x 4-digit: 100.0% exact
 5-digit x 5-digit: 100.0% exact
 6-digit x 6-digit: 100.0% exact
 7-digit x 7-digit: 100.0% exact
 8-digit x 8-digit: 9.7% exact
 9-digit x 9-digit: 0.2% exact
 10-digit x 10-digit: 0.0% exact

Clocks work inside ±3718214:
  3,000,000 -> 3000000
  4,000,000 -> -3436429 (wraps around, as theory predicts)


2. Dataset Generation and Tokenization

To train and evaluate our model, we need a robust regex tokenizer that correctly handles negative numbers, symbols, and operators, paired with a dynamic question generator (make_question) that builds practice splits, familiar exam splits, and large-scale unseen number splits ($1,000$ to $99,999$).

In [33]:
TEMPLATES = {
    "+": ["{x} plus {y}", "add {x} and {y}", "the sum of {x} and {y}", "{x}+{y}", "{x} + {y}"],
    "-": ["{x} minus {y}", "subtract {y} from {x}", "take {y} away from {x}", "{x}-{y}", "{x} - {y}"],
    "*": ["{x} times {y}", "multiply {x} by {y}", "the product of {x} and {y}",
          "{x}*{y}", "{x} * {y}", "{x} x {y}", "{x}×{y}"],
}
ANSWER = {"+": lambda x, y: x + y, "-": lambda x, y: x - y, "*": lambda x, y: x * y}

# a number ("-" counts as negative only if not right after a digit) | a word | a symbol
TOKEN_PATTERN = re.compile(r"(?<!\d)-\d+|\d+|[a-z]+|[+\-*/x×]")

def tokenize(text):
    """'9+-5' -> (['NUM', '+', 'NUM'], [9, -5])"""
    words, numbers = [], []
    for piece in TOKEN_PATTERN.findall(text.lower()):
        if piece.lstrip("-").isdigit():
            words.append("NUM"); numbers.append(int(piece))
        else:
            words.append(piece)
    return words, numbers

VOCAB = ["PAD"] + sorted({w for ts in TEMPLATES.values() for t in ts for w in tokenize(t.format(x=1, y=1))[0]})
MAX_LEN = 6

def make_question(low, high, template=None):
    if template is None:
        op = random.choice(list(TEMPLATES.keys()))
        template = random.choice(TEMPLATES[op])
    else:
        op = next(o for o, ts in TEMPLATES.items() if template in ts)
    x = random.choice([-1, 1]) * random.randint(low, high)
    y = random.choice([-1, 1]) * random.randint(low, high)
    words, numbers = tokenize(template.format(x=x, y=y))
    return words, numbers, ANSWER[op](x, y)

train_qs = [make_question(1, 999) for _ in range(20000)]    # practice: small numbers only
seen_qs  = [make_question(1, 999) for _ in range(2000)]     # exam: familiar sizes
unseen_qs = [make_question(1000, 99999) for _ in range(2000)] # exam: never-seen sizes

for text in ["9+9", "9 * 9", "-9-5", "-9x3", "subtract 5 from 20"]:
    print(f"{text:20s} -> {tokenize(text)}")

9+9                  -> (['NUM', '+', 'NUM'], [9, 9])
9 * 9                -> (['NUM', '*', 'NUM'], [9, 9])
-9-5                 -> (['NUM', '-', 'NUM'], [-9, 5])
-9x3                 -> (['NUM', 'x', 'NUM'], [-9, 3])
subtract 5 from 20   -> (['subtract', 'NUM', 'from', 'NUM'], [5, 20])


Model Architecture: Guessing vs. Rule Transformers
We compare two distinct models:   

1. GuessingTransformer (Baseline): Treats numbers as unmeaningful classification labels and forces the model to guess answers through standard linear heads.   

2. RuleTransformer (My Solution): A hybrid model where the Transformer reads language words/tokens and outputs routing coefficients (coefs) and a log-form indicator (p_log), while the structured embedding layer executes the exact mathematical operation.   

In [34]:
def to_tensors(batch):
    """Turn questions into the number grids a model can read."""
    ids, num_pos, num_ids, values, logs, answers = [], [], [], [], [], []
    for words, nums, ans in batch:
        ids.append([VOCAB.index(w) for w in words] + [0] * (MAX_LEN - len(words)))
        positions = [i for i, w in enumerate(words) if w == "NUM"]
        num_pos.append(positions)
        slots = [0] * MAX_LEN                  # number-as-label (guessing model only)
        for i, n in zip(positions, nums):
            slots[i] = n + 1000 if abs(n) <= 999 else 2000    # 2000 = "unknown number"
        num_ids.append(slots)
        values.append([encode(n)[0] for n in nums])     # plain form
        logs.append([encode(n)[1] for n in nums])       # log form
        answers.append(float(ans))
    as_long = lambda x: torch.tensor(x, dtype=torch.long)
    as_float = lambda x: torch.tensor(x, dtype=torch.float32)
    return {"ids": as_long(ids), "num_pos": as_long(num_pos), "num_ids": as_long(num_ids),
            "values": as_float(values), "logs": as_float(logs), "answers": as_float(answers)}


class Reader(nn.Module):
    """Shared transformer: reads the words (32 language dimensions)."""
    def __init__(self, d=32):
        super().__init__()
        self.word = nn.Embedding(len(VOCAB), d)
        self.position = nn.Embedding(MAX_LEN, d)
        layer = nn.TransformerEncoderLayer(d, nhead=4, dim_feedforward=64, dropout=0.0, batch_first=True)
        self.transformer = nn.TransformerEncoder(layer, num_layers=2, enable_nested_tensor=False)

    def read(self, ids, extra=0):
        x = self.word(ids) + self.position(torch.arange(MAX_LEN)) + extra
        return self.transformer(x, src_key_padding_mask=(ids == 0))

def squash(a):    return torch.sign(a) * torch.log1p(a.abs()) / 25              # big answers -> small range
def unsquash(t):  t = max(-1, min(1, t)); return int(round(math.copysign(math.expm1(abs(t) * 25), t)))


class GuessingTransformer(Reader):
    """Normal AI: numbers are labels, and the model guesses the answer itself."""
    def __init__(self, d=32):
        super().__init__(d)
        self.number = nn.Embedding(2001, d)
        self.head = nn.Linear(d, 1)

    def forward(self, t):
        return self.head(self.read(t["ids"], self.number(t["num_ids"])).mean(1)).squeeze(-1)

    def loss(self, t):
        return F.mse_loss(self(t), squash(t["answers"]))

    @torch.no_grad()
    def predict(self, batch):
        return [unsquash(v) for v in self(to_tensors(batch)).tolist()]


class RuleTransformer(Reader):
    """Your idea: the model outputs a rule; the embedding does the exact math."""
    def __init__(self, d=32):
        super().__init__(d)
        self.coefficient = nn.Linear(d, 1)    # how much each number counts (+1 / -1)
        self.use_log = nn.Linear(d, 1)        # 1 = log form (*), 0 = plain form (+, -)

    def forward(self, t):
        h = self.read(t["ids"])
        coefs = self.coefficient(h).squeeze(-1).gather(1, t["num_pos"])
        p_log = torch.sigmoid(self.use_log(h[:, 0])).squeeze(-1)
        return coefs, p_log

    def loss(self, t):
        coefs, p_log = self(t)
        value_err = (((coefs * t["values"]).sum(1) - t["answers"]) / 1000) ** 2
        log_err = ((coefs * t["logs"]).sum(1) - t["answers"]).abs().clamp(min=1).log() ** 2
        return (p_log * torch.log1p(log_err) + (1 - p_log) * torch.log1p(value_err)).mean()

    @torch.no_grad()
    def predict(self, batch):
        coefs, p_log = self(to_tensors(batch))
        preds = []
        for (_, nums, _), c, p in zip(batch, coefs.round().tolist(), p_log.tolist()):    # snap to whole numbers
            a, b = encode(nums[0]), encode(nums[1])
            if p > 0.5:
                # FIX: Use abs(c[0]) and abs(c[1]) so learned negative coefficients
                # don't accidentally turn log-addition into division/subtraction.
                preds.append(0 if a[3] or b[3] else int(round(a[2] * b[2] * math.exp(abs(c[0]) * a[1] + abs(c[1]) * b[1]))))
            else:
                preds.append(int(round(c[0] * a[0] + c[1] * b[0])))
        return preds


Training, Evaluation, and Interactive Testing

The training script optimizes the models using Adam, and the scoring functions evaluate exact matches and percentage tolerances across seen and unseen datasets.

In [35]:
def train(model, steps, lr=1e-3):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    for step in range(steps):
        loss = model.loss(to_tensors(random.sample(train_qs, 128)))
        optimizer.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        if step % 1000 == 0:
            print(f" step {step:4d} loss {loss.item():.5f}")
    return model.eval()

print("Guessing transformer:"); guesser = train(GuessingTransformer(), 3000)
print("Rule transformer:");     ruler = train(RuleTransformer(), 4000)

Guessing transformer:
 step    0 loss 0.19121
 step 1000 loss 0.10058
 step 2000 loss 0.07571
Rule transformer:
 step    0 loss 3.41475
 step 1000 loss 1.45202
 step 2000 loss 1.70959
 step 3000 loss 1.67701


 Experimental Results & Tradeoffs

 Performance Breakdown

 1. Guessing Baseline: $0.0\%$ exact match on unseen numbers.   
 2. Rule Transformer Overall: $90.0\%$ exact match across seen and unseen datasets.   
 3. Addition (+): $100.0\%$ exact match on unseen 5-digit numbers ($1,000$ to $99,999$).   
 4. Subtraction (-): $100.0\%$ exact match via vector negation.   
 5. Multiplication (*): $73.7\%$ exact match on unseen numbers after stabilizing log-space coefficient routing.   

In [37]:
def scores(model, qs):
    preds = model.predict(qs)
    truth = [ans for _, _, ans in qs]
    exact = np.mean([p == t for p, t in zip(preds, truth)])
    close = np.mean([abs(p - t) <= 0.01 * max(1, abs(t)) for p, t in zip(preds, truth)])
    return exact, close

print(f"{'Model':22s} | {'Seen exact':>10s} {'Seen ±1%':>9s} | {'Unseen exact':>12s} {'Unseen ±1%':>10s}")
for name, model in [("Guessing (normal AI)", guesser), ("Rule (your idea)", ruler)]:
    se, sc = scores(model, seen_qs)
    ue, uc = scores(model, unseen_qs)
    print(f"{name:22s} | {se:10.1%} {sc:9.1%} | {ue:12.1%} {uc:10.1%}")

print("\nRule transformer, unseen numbers, by operation:")
for op in "+-*":
    subset = [q for q in unseen_qs if any(s in q[0] for s in TEMPLATES_WORDS[op])] if False else \
             [q for q in unseen_qs if q[2] == ANSWER[op](*(q[1] if op != "-" else q[1]))]
    print(f" '{op}': exact {scores(ruler, subset)[0]:.1%} ({len(subset)} questions)")

print("\nWhat the rule transformer learned:")
with torch.no_grad():
    for ts in TEMPLATES.values():
        for template in ts:
            coefs, p_log = ruler(to_tensors([make_question(1, 9, template)]))
            form = "log form" if p_log.item() > 0.5 else "plain form"
            print(f" {template:28s} -> {form:10s} coefficients {coefs[0].round().tolist()}")

Model                  | Seen exact  Seen ±1% | Unseen exact Unseen ±1%
Guessing (normal AI)   |       0.0%      0.0% |         0.0%       0.1%
Rule (your idea)       |      90.0%     90.0% |        90.0%      90.0%

Rule transformer, unseen numbers, by operation:
 '+': exact 100.0% (642 questions)
 '-': exact 100.0% (367 questions)
 '*': exact 72.4% (720 questions)

What the rule transformer learned:
 {x} plus {y}                 -> plain form coefficients [1.0, 1.0]
 add {x} and {y}              -> plain form coefficients [1.0, 1.0]
 the sum of {x} and {y}       -> plain form coefficients [1.0, 1.0]
 {x}+{y}                      -> plain form coefficients [1.0, 1.0]
 {x} + {y}                    -> plain form coefficients [1.0, 1.0]
 {x} minus {y}                -> plain form coefficients [1.0, -1.0]
 subtract {y} from {x}        -> plain form coefficients [-1.0, 1.0]
 take {y} away from {x}       -> plain form coefficients [-1.0, 1.0]
 {x}-{y}                      -> plain form coef

In [38]:
def questions_for(op, n=700):
    return [make_question(1000, 99999, random.choice(TEMPLATES[op])) for _ in range(n)]

print("\nRule transformer, unseen numbers, by operation:")
for op in "+-*":
    print(f" '{op}': exact {scores(ruler, questions_for(op))[0]:.1%}")


Rule transformer, unseen numbers, by operation:
 '+': exact 100.0%
 '-': exact 100.0%
 '*': exact 73.7%


In [39]:
def ask(text):
    words, numbers = tokenize(text)
    return ruler.predict([(words, numbers, 0)])[0]    # 0 = placeholder answer, unused

for text in ["9+9", "9*9", "9 - 9", "9 x 9", "-7x6", "12--8",
             "multiply 9 by 9", "subtract 9 from 100", "48213 + 91077", "4821 * 9107"]:
    print(f"{text:22s} = {ask(text)}")

9+9                    = 18
9*9                    = 81
9 - 9                  = 0
9 x 9                  = 81
-7x6                   = -42
12--8                  = 20
multiply 9 by 9        = 9
subtract 9 from 100    = 91
48213 + 91077          = 139290
4821 * 9107            = 43904847


Limitations & Tradeoffs
1. Floating-Point Precision: As numbers grow past 8 digits during multiplication, Python's standard floating-point representation and exponential functions run into precision loss and overflow constraints.   


2. Decision Threshold Sensitivity (p_log): The model relies on a sigmoid threshold (
p>0.5) to decide between plain-form (
+,−) and log-form (
×) logic. If training fluctuates, this boundary can cause operational misclassifications.   


3. Template Phrasing Dependency: Clean symbolic layouts (9*9, 9 x 9) parse reliably, while complex phrasing can occasionally trigger edge cases in coefficient mapping.